In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    classification_report, confusion_matrix
)
import xgboost as xgb
import joblib
import json
import os

# Set display options
pd.set_option('display.max_columns', None)

# Set plot style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [3]:
print("💓 Processing validation vitals from raw CHARTEVENTS...")
print("=" * 60)
print("⏳ This will take 20-30 minutes (processing ~330M rows)")
print()

# Check if already processed
val_vitals_path = '../../data/processed/vital_features_val.csv'

if os.path.exists(val_vitals_path):
    # Check if it's valid (has patients with vitals)
    test_df = pd.read_csv(val_vitals_path)
    has_vitals = (test_df.iloc[:, 1:].sum(axis=1) > 0).sum()
    
    if has_vitals > 0:
        print(f"✅ Found existing VALID validation vitals!")
        vital_features_val = test_df
        print(f"   Shape: {vital_features_val.shape}")
        print(f"   Patients with vitals: {has_vitals:,}")
    else:
        print(f"⚠️  Existing file is empty (0 patients with vitals)")
        print(f"   Need to re-process from CHARTEVENTS...")
        os.path.exists = lambda x: False
        
if not os.path.exists(val_vitals_path):
    print(f"⚠️  Processing validation vitals from CHARTEVENTS.csv...")
    
    # Load validation patient IDs
    val_patients = pd.read_csv('../../data/processed/val_patients.csv')
    val_patient_ids = val_patients['SUBJECT_ID'].values
    
    print(f"   Validation patients: {len(val_patient_ids):,}")
    
    # Important vital codes
    important_vital_codes = [220045, 220050, 220051, 220052, 220179, 220180, 
                            220181, 220210, 220277, 223761, 223762, 220339, 223835]
    
    # Load CHARTEVENTS in chunks
    chunk_size = 1_000_000
    vital_chunks = []
    
    chunk_counter = 0
    total_rows_processed = 0
    rows_kept = 0
    
    for chunk in tqdm(pd.read_csv('../../data/raw/CHARTEVENTS.csv', chunksize=chunk_size)):
        chunk_counter += 1
        total_rows_processed += len(chunk)
        
        # Filter to validation patients + important vitals
        chunk_filtered = chunk[
            (chunk['SUBJECT_ID'].isin(val_patient_ids)) &
            (chunk['ITEMID'].isin(important_vital_codes)) &
            (chunk['VALUENUM'].notna())
        ]
        
        rows_kept += len(chunk_filtered)
        
        if len(chunk_filtered) > 0:
            vital_chunks.append(chunk_filtered)
        
        # Print progress every 10 chunks
        if chunk_counter % 10 == 0:
            print(f"  Processed {total_rows_processed:,} rows, kept {rows_kept:,} ({rows_kept/total_rows_processed*100:.2f}%)")
    
    # Combine chunks
    if vital_chunks:
        print(f"\n🔗 Combining {len(vital_chunks)} chunks...")
        vitals_val_raw = pd.concat(vital_chunks, ignore_index=True)
        
        print(f"✅ Filtered vitals loaded!")
        print(f"   Total rows: {total_rows_processed:,}")
        print(f"   Kept: {len(vitals_val_raw):,}")
        print(f"   Unique patients: {vitals_val_raw['SUBJECT_ID'].nunique():,}")
        
        # Aggregate features per patient
        print(f"\n⏱️  Aggregating features for {len(val_patient_ids):,} patients...")
        
        agg_features = []
        
        for patient_id in tqdm(val_patient_ids, desc="Aggregating patients"):
            patient_vitals = vitals_val_raw[vitals_val_raw['SUBJECT_ID'] == patient_id]
            
            if len(patient_vitals) == 0:
                # No vitals for this patient
                features = {'SUBJECT_ID': patient_id}
                for itemid in important_vital_codes:
                    features[f'vital_{itemid}_mean'] = np.nan
                    features[f'vital_{itemid}_min'] = np.nan
                    features[f'vital_{itemid}_max'] = np.nan
                    features[f'vital_{itemid}_std'] = np.nan
                    features[f'vital_{itemid}_count'] = 0
                agg_features.append(features)
                continue
            
            features = {'SUBJECT_ID': patient_id}
            
            for itemid in important_vital_codes:
                vital_values = patient_vitals[patient_vitals['ITEMID'] == itemid]['VALUENUM'].dropna()
                
                if len(vital_values) > 0:
                    features[f'vital_{itemid}_mean'] = vital_values.mean()
                    features[f'vital_{itemid}_min'] = vital_values.min()
                    features[f'vital_{itemid}_max'] = vital_values.max()
                    features[f'vital_{itemid}_std'] = vital_values.std() if len(vital_values) > 1 else 0
                    features[f'vital_{itemid}_count'] = len(vital_values)
                else:
                    features[f'vital_{itemid}_mean'] = np.nan
                    features[f'vital_{itemid}_min'] = np.nan
                    features[f'vital_{itemid}_max'] = np.nan
                    features[f'vital_{itemid}_std'] = np.nan
                    features[f'vital_{itemid}_count'] = 0
            
            agg_features.append(features)
        
        vital_features_val = pd.DataFrame(agg_features)
        
        # Save
        vital_features_val.to_csv(val_vitals_path, index=False)
        print(f"\n✅ Saved to: {val_vitals_path}")
    else:
        print(f"\n⚠️  WARNING: No vital measurements found for validation patients!")
        print(f"   This means validation patients have no CHARTEVENTS data.")
        print(f"   Creating empty feature file...")
        
        # Create empty features
        features_list = []
        for patient_id in val_patient_ids:
            features = {'SUBJECT_ID': patient_id}
            for itemid in important_vital_codes:
                features[f'vital_{itemid}_mean'] = np.nan
                features[f'vital_{itemid}_min'] = np.nan
                features[f'vital_{itemid}_max'] = np.nan
                features[f'vital_{itemid}_std'] = np.nan
                features[f'vital_{itemid}_count'] = 0
            features_list.append(features)
        
        vital_features_val = pd.DataFrame(features_list)
        vital_features_val.to_csv(val_vitals_path, index=False)

print(f"\n✅ Validation vitals ready!")
print(f"   Shape: {vital_features_val.shape}")
print(f"   Patients: {len(vital_features_val):,}")

# Check coverage
has_vitals = (vital_features_val.iloc[:, 1:].sum(axis=1) > 0).sum()
coverage_pct = has_vitals / len(vital_features_val) * 100
print(f"   Patients with vitals: {has_vitals:,} ({coverage_pct:.1f}%)")



💓 Processing validation vitals from raw CHARTEVENTS...
⏳ This will take 20-30 minutes (processing ~330M rows)

⚠️  Existing file is empty (0 patients with vitals)
   Need to re-process from CHARTEVENTS...
⚠️  Processing validation vitals from CHARTEVENTS.csv...
   Validation patients: 4,675

📊 Processing CHARTEVENTS in chunks (this takes 20-30 min)...
☕ Perfect time for a coffee break!



10it [00:12,  1.15s/it]

  Processed 10,000,000 rows, kept 1,004,136 (10.04%)


20it [00:23,  1.10s/it]

  Processed 20,000,000 rows, kept 1,610,363 (8.05%)


...

  Processed 330,000,000 rows, kept 2,521,790 (0.76%)


331it [06:24,  1.16s/it]



🔗 Combining 35 chunks...
✅ Filtered vitals loaded!
   Total rows: 330,712,483
   Kept: 2,521,790
   Unique patients: 2,160

⏱️  Aggregating features for 4,675 patients...


Aggregating patients: 100%|███████████████████████████████████████████████████████| 4675/4675 [00:23<00:00, 196.51it/s]



✅ Saved to: ../data/processed/vital_features_val.csv

✅ Validation vitals ready!
   Shape: (4675, 66)
   Patients: 4,675
   Patients with vitals: 2,160 (46.2%)


In [4]:
print("🎯 Loading validation labels...")
print("=" * 60)

# Load labels
labels = pd.read_csv('../../data/processed/patient_multilabels.csv')

# Filter to validation patients
val_labels = labels[labels['SUBJECT_ID'].isin(vital_features_val['SUBJECT_ID'])].copy()

print(f"✅ Validation labels loaded!")
print(f"   Patients: {len(val_labels):,}")

# Disease columns
disease_cols = [
    'SEPSIS', 'PNEUMONIA', 'RESPIRATORY_FAILURE', 
    'ACUTE_KIDNEY_INJURY', 'HEART_FAILURE', 
    'ATRIAL_FIBRILLATION', 'CORONARY_ARTERY_DISEASE', 
    'ANEMIA', 'PANCREATITIS'
]

print(f"\nValidation disease distribution:")
for disease in disease_cols:
    count = val_labels[disease].sum()
    pct = count / len(val_labels) * 100
    print(f"  {disease:30s}: {count:5,} ({pct:5.1f}%)")

🎯 Loading validation labels...
✅ Validation labels loaded!
   Patients: 4,675

Validation disease distribution:
  SEPSIS                        :   880 ( 18.8%)
  PNEUMONIA                     :   987 ( 21.1%)
  RESPIRATORY_FAILURE           : 1,713 ( 36.6%)
  ACUTE_KIDNEY_INJURY           : 1,750 ( 37.4%)
  HEART_FAILURE                 : 1,559 ( 33.3%)
  ATRIAL_FIBRILLATION           : 1,562 ( 33.4%)
  CORONARY_ARTERY_DISEASE       : 1,784 ( 38.2%)
  ANEMIA                        : 1,618 ( 34.6%)
  PANCREATITIS                  :   675 ( 14.4%)


In [5]:
print("🔗 Merging validation features with labels...")
print("=" * 60)

# Merge
val_data = vital_features_val.merge(val_labels, on='SUBJECT_ID', how='inner')

print(f"✅ Merged successfully!")
print(f"   Shape: {val_data.shape}")
print(f"   Patients: {len(val_data):,}")

# Separate features and labels
X_val = val_data.drop(columns=['SUBJECT_ID'] + disease_cols)
y_val = val_data[disease_cols]
patient_ids_val = val_data['SUBJECT_ID']

print(f"\n📊 Validation dataset:")
print(f"   X_val (features): {X_val.shape}")
print(f"   y_val (labels):   {y_val.shape}")

🔗 Merging validation features with labels...
✅ Merged successfully!
   Shape: (4675, 76)
   Patients: 4,675

📊 Validation dataset:
   X_val (features): (4675, 66)
   y_val (labels):   (4675, 9)


In [6]:
print("🔧 Loading preprocessing objects...")
print("=" * 60)

# Load imputer, scaler, and preprocessing info
imputer = joblib.load('../../models/agent3_imputer.joblib')
scaler = joblib.load('../../models/agent3_scaler.joblib')

with open('../../models/agent3_preprocessing_info.json', 'r') as f:
    preprocessing_info = json.load(f)

print(f"✅ Loaded preprocessing objects!")
print(f"   Features to use: {preprocessing_info['n_features']}")
print(f"   Features dropped: {len(preprocessing_info['features_dropped'])}")

🔧 Loading preprocessing objects...
✅ Loaded preprocessing objects!
   Features to use: 62
   Features dropped: 4


In [7]:
print("🔧 Applying preprocessing to validation data...")
print("=" * 60)

# Step 1: Drop same features as training
features_to_drop = preprocessing_info['features_dropped']
X_val_reduced = X_val.drop(columns=features_to_drop, errors='ignore')

print(f"Step 1: Drop features")
print(f"  Features before: {X_val.shape[1]}")
print(f"  Features after:  {X_val_reduced.shape[1]}")

# Check for feature alignment
expected_features = preprocessing_info['features_used']
current_features = X_val_reduced.columns.tolist()

# Handle missing/extra features
missing = set(expected_features) - set(current_features)
extra = set(current_features) - set(expected_features)

if missing:
    print(f"\n  Adding {len(missing)} missing features with NaN")
    for feat in missing:
        X_val_reduced[feat] = np.nan

if extra:
    print(f"  Dropping {len(extra)} extra features")
    X_val_reduced = X_val_reduced.drop(columns=list(extra))

# Reorder to match training
X_val_reduced = X_val_reduced[expected_features]

# Impute
X_val_imputed = pd.DataFrame(
    imputer.transform(X_val_reduced),
    columns=X_val_reduced.columns,
    index=X_val_reduced.index
)

print(f"\nStep 2: Impute missing values")
print(f"  Missing remaining: {X_val_imputed.isnull().sum().sum()}")

# Scale
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val_imputed),
    columns=X_val_imputed.columns,
    index=X_val_imputed.index
)

print(f"\nStep 3: Normalize")
print(f"  Mean: {X_val_scaled.mean().mean():.6f}")
print(f"  Std:  {X_val_scaled.std().mean():.6f}")

print(f"\n✅ Preprocessing complete!")
print(f"   Final shape: {X_val_scaled.shape}")

🔧 Applying preprocessing to validation data...
Step 1: Drop features
  Features before: 66
  Features after:  62

Step 2: Impute missing values
  Missing remaining: 0

Step 3: Normalize
  Mean: 0.003767
  Std:  0.950495

✅ Preprocessing complete!
   Final shape: (4675, 62)


In [8]:
print("🤖 Loading Agent 3 models and evaluating...")
print("=" * 60)

# Load models
models = {}
for disease in disease_cols:
    model_path = f'../../models/agent3_{disease.lower()}.joblib'
    models[disease] = joblib.load(model_path)
    print(f"  ✅ Loaded: {disease}")

print(f"\n📊 Evaluating on validation set...")

val_predictions = {}
val_metrics = {}

for disease in tqdm(disease_cols, desc="Evaluating"):
    # Get true labels
    y_true = y_val[disease].values
    
    # Predict
    y_pred_proba = models[disease].predict_proba(X_val_scaled)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred_proba)
    
    # Store
    val_predictions[disease] = y_pred_proba
    val_metrics[disease] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'auc_roc': auc,
        'positive_samples': int(y_true.sum()),
        'prevalence': float(y_true.mean())
    }
    
    print(f"{disease:30s}: Acc={accuracy:.3f}, F1={f1:.3f}, AUC={auc:.3f}")

print(f"\n✅ Validation complete!")

🤖 Loading Agent 3 models and evaluating...
  ✅ Loaded: SEPSIS
  ✅ Loaded: PNEUMONIA
  ✅ Loaded: RESPIRATORY_FAILURE
  ✅ Loaded: ACUTE_KIDNEY_INJURY
  ✅ Loaded: HEART_FAILURE
  ✅ Loaded: ATRIAL_FIBRILLATION
  ✅ Loaded: CORONARY_ARTERY_DISEASE
  ✅ Loaded: ANEMIA
  ✅ Loaded: PANCREATITIS

📊 Evaluating on validation set...


Evaluating:  22%|████████████████                                                        | 2/9 [00:00<00:00, 17.80it/s]

SEPSIS                        : Acc=0.846, F1=0.459, AUC=0.828
PNEUMONIA                     : Acc=0.827, F1=0.424, AUC=0.825
RESPIRATORY_FAILURE           : Acc=0.781, F1=0.664, AUC=0.828
ACUTE_KIDNEY_INJURY           : Acc=0.782, F1=0.684, AUC=0.842


Evaluating: 100%|████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 38.00it/s]

HEART_FAILURE                 : Acc=0.781, F1=0.622, AUC=0.843
ATRIAL_FIBRILLATION           : Acc=0.721, F1=0.523, AUC=0.756
CORONARY_ARTERY_DISEASE       : Acc=0.673, F1=0.377, AUC=0.687
ANEMIA                        : Acc=0.713, F1=0.475, AUC=0.726
PANCREATITIS                  : Acc=0.859, F1=0.134, AUC=0.732

✅ Validation complete!


In [9]:
print("📊 Agent 3 (Vitals) Validation Results")
print("=" * 80)

# Create metrics dataframe
val_metrics_df = pd.DataFrame(val_metrics).T
val_metrics_df = val_metrics_df.round(3)

print(val_metrics_df.to_string())

print(f"\n{'='*80}")
print(f"📈 Summary:")
print(f"   Average Accuracy: {val_metrics_df['accuracy'].mean():.3f} ± {val_metrics_df['accuracy'].std():.3f}")
print(f"   Average F1 Score: {val_metrics_df['f1_score'].mean():.3f} ± {val_metrics_df['f1_score'].std():.3f}")
print(f"   Average AUC-ROC:  {val_metrics_df['auc_roc'].mean():.3f} ± {val_metrics_df['auc_roc'].std():.3f}")

# Save metrics
val_metrics_df.to_csv('../../results/agent3_val_metrics.csv')
print(f"\n✅ Saved: results/agent3_val_metrics.csv")

📊 Agent 3 (Vitals) Validation Results
                         accuracy  f1_score  auc_roc  positive_samples  prevalence
SEPSIS                      0.846     0.459    0.828             880.0       0.188
PNEUMONIA                   0.827     0.424    0.825             987.0       0.211
RESPIRATORY_FAILURE         0.781     0.664    0.828            1713.0       0.366
ACUTE_KIDNEY_INJURY         0.782     0.684    0.842            1750.0       0.374
HEART_FAILURE               0.781     0.622    0.843            1559.0       0.333
ATRIAL_FIBRILLATION         0.721     0.523    0.756            1562.0       0.334
CORONARY_ARTERY_DISEASE     0.673     0.377    0.687            1784.0       0.382
ANEMIA                      0.713     0.475    0.726            1618.0       0.346
PANCREATITIS                0.859     0.134    0.732             675.0       0.144

📈 Summary:
   Average Accuracy: 0.776 ± 0.063
   Average F1 Score: 0.485 ± 0.170
   Average AUC-ROC:  0.785 ± 0.060

✅ Saved: resul

In [10]:
print("📊 Agent 3: Training vs Validation Comparison")
print("=" * 80)

# Load training metrics
train_metrics_df = pd.read_csv('../../results/agent3_train_metrics.csv', index_col=0)

# Compare
comparison = pd.DataFrame({
    'Train_AUC': train_metrics_df['auc_roc'],
    'Val_AUC': val_metrics_df['auc_roc'],
    'AUC_Drop': train_metrics_df['auc_roc'] - val_metrics_df['auc_roc'],
    'Train_F1': train_metrics_df['f1_score'],
    'Val_F1': val_metrics_df['f1_score']
})

comparison = comparison.round(3)
print(comparison.to_string())

print(f"\n{'='*80}")
print(f"📈 Agent 3 Overfitting Analysis:")
print(f"   Training AUC:   {train_metrics_df['auc_roc'].mean():.3f}")
print(f"   Validation AUC: {val_metrics_df['auc_roc'].mean():.3f}")
print(f"   Average drop:   {comparison['AUC_Drop'].mean():.3f} ({comparison['AUC_Drop'].mean()*100:.1f}%)")

if comparison['AUC_Drop'].mean() < 0.05:
    print(f"\n   ✅ EXCELLENT: Minimal overfitting!")
elif comparison['AUC_Drop'].mean() < 0.10:
    print(f"\n   ✅ GOOD: Acceptable generalization")
else:
    print(f"\n   ⚠️  WARNING: Some overfitting detected")

comparison.to_csv('../../results/agent3_train_val_comparison.csv')
print(f"\n✅ Saved: results/agent3_train_val_comparison.csv")

📊 Agent 3: Training vs Validation Comparison
                         Train_AUC  Val_AUC  AUC_Drop  Train_F1  Val_F1
SEPSIS                       0.898    0.828     0.070     0.611   0.459
PNEUMONIA                    0.882    0.825     0.057     0.573   0.424
RESPIRATORY_FAILURE          0.887    0.828     0.059     0.722   0.664
ACUTE_KIDNEY_INJURY          0.896    0.842     0.054     0.761   0.684
HEART_FAILURE                0.900    0.843     0.057     0.701   0.622
ATRIAL_FIBRILLATION          0.857    0.756     0.101     0.660   0.523
CORONARY_ARTERY_DISEASE      0.794    0.687     0.107     0.512   0.377
ANEMIA                       0.830    0.726     0.104     0.635   0.475
PANCREATITIS                 0.859    0.732     0.127     0.389   0.134

📈 Agent 3 Overfitting Analysis:
   Training AUC:   0.867
   Validation AUC: 0.785
   Average drop:   0.082 (8.2%)

   ✅ GOOD: Acceptable generalization

✅ Saved: results/agent3_train_val_comparison.csv
